In [1]:
import matplotlib
import matplotlib.style as mplstyle
matplotlib.use('TkAGG')
mplstyle.use('fast')

matplotlib.rcParams['path.simplify'] = True
matplotlib.rcParams['path.simplify_threshold'] = 1.0
matplotlib.rcParams['agg.path.chunksize'] = 10000

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
from matplotlib.gridspec import GridSpec
from matplotlib.gridspec import GridSpec,GridSpecFromSubplotSpec
import matplotlib.image as mpimg
from mpl_toolkits.axes_grid1 import make_axes_locatable

from urllib.parse import urlencode
from urllib.request import urlretrieve

import numpy as np
import numpy.core.defchararray as np_f
import pandas as pd
import scipy as sci
from scipy.stats import f
from scipy.stats import kde
from subprocess import *
import os
import glob
from pathlib import Path
import re

from astropy.table import Table
from astropy import constants as const
from astropy import units as u
from astropy.io import fits
from astropy import coordinates as coords

import importlib
from tqdm.notebook import trange, tqdm

import time

import warnings

import ResearchTools.LCtools as LCtools
import VarStar_Vi_plot_functions as vi

from multiprocessing import Pool

In [2]:
spec_dir = "/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/HARD_COPY_ORGINAL_DATA/SDSS_spec/02-26-2020/SDSSspec/"
CSS_LC_dir = "/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/HARD_COPY_ORGINAL_DATA/CSS_LCs/csvs/"
ZTF_LC_dir = "/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/HARD_COPY_ORGINAL_DATA/ZTF/DATA/07-27-2021/"

ZTF_filters = ['g', 'r']
ZTF_LC_file_names = [f'TDSS_VarStar_ZTFDR6_{ZTF_filter}_GroupID.fits' for ZTF_filter in ZTF_filters]
ZTF_g_LCs = Table.read(ZTF_LC_dir + ZTF_LC_file_names[0])
ZTF_r_LCs = Table.read(ZTF_LC_dir + ZTF_LC_file_names[1])

In [3]:
prop_out_dir, CSS_LC_plot_dir, ZTF_LC_plot_dir, Vi_plots_dir, datestr = vi.makeViDirs()

In [4]:
nbins = 50
TDSSprop = vi.TDSSprop(nbins)

TDSS_Table = Table.read('/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Statistical_Testing/LC_stat_calculations/Control_StatsCalc/TDSS_VarStar_FINAL_Var_ALL_PROP_STATS_03-15-2022.fits')

xiyizi = [None, None, None]

for ii, best_LC in enumerate(['CSS', 'ZTF_g', 'ZTF_r']):
    all_is_periodic = TDSS_Table[best_LC + '_logProb'] <= -5
    xiyizi[ii] = vi.make_middle_heatmap(TDSS_Table, best_LC, all_is_periodic)

In [6]:
def make_Vi_propID(prop_id, TDSS_Table, TDSSprop, logFAP_limit=-5):
    ROW = TDSS_Table[prop_id]
    object_ra = ROW['ra_GaiaEDR3']
    object_dec = ROW['dec_GaiaEDR3']
    ra_string = '{:0>9.5f}'.format(object_ra)
    dec_string = '{:0=+9.5f}'.format(object_dec)

    this_Viplot_filename = f"{prop_id}_{ra_string}{dec_string}_Vi.pdf"
    
    mjd = ROW['mjd']
    plate = ROW['plate']
    fiberid = ROW['fiber']
    mjd_string = '{:0>5}'.format(str(np.int64(mjd)))
    plate_string = '{:0>4}'.format(str(np.int64(plate)))
    fiberid_string = '{:0>4}'.format(str(np.int64(fiberid)))
    long_filename = f"spec-{plate_string}-{mjd_string}-{fiberid_string}.fits"

    fig = plt.figure(figsize=(13, 9), constrained_layout=True)
    gs = GridSpec(2, 6, figure=fig, height_ratios=[1, 1], width_ratios=[1, 1, 1, 1, 1, 1])  # , hspace=0.3, wspace=0.5)
    ax1 = fig.add_subplot(gs[0, :2])  # LC
    ax2 = fig.add_subplot(gs[0, 2:4])  # Chi2 vs Amp plots OR Palerversa+2013 plot
    ax3 = fig.add_subplot(gs[0, 4:])  # CMD
    ax4 = fig.add_subplot(gs[1, :])  # spectra with lines
    fig.suptitle(f'RA= {ra_string} DEC= {dec_string}', fontsize=16)

    best_LC = vi.find_best_LC(ROW, logFAP_limit)

    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax1, best_LC, CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False)
    vi.plot_middle(ROW, TDSS_Table, best_LC, xiyizi, ax2, log10FAP=logFAP_limit)
    vi.plot_CMD(TDSSprop, prop_id, ax3)
    this_EqW = vi.plot_SDSSspec(ROW, spec_dir, ax4)

    return fig, [ax1, ax2, ax3, ax4], this_Viplot_filename
    

def make_Vi_propID_parallel(prop_id):
    #TDSS_Table
    #TDSSprop,
    logFAP_limit=-5
    
    ROW = TDSS_Table[prop_id]
    object_ra = ROW['ra_GaiaEDR3']
    object_dec = ROW['dec_GaiaEDR3']
    ra_string = '{:0>9.5f}'.format(object_ra)
    dec_string = '{:0=+9.5f}'.format(object_dec)

    this_Viplot_filename = f"{prop_id}_{ra_string}{dec_string}_Vi.pdf"
    
    mjd = ROW['mjd']
    plate = ROW['plate']
    fiberid = ROW['fiber']
    mjd_string = '{:0>5}'.format(str(np.int64(mjd)))
    plate_string = '{:0>4}'.format(str(np.int64(plate)))
    fiberid_string = '{:0>4}'.format(str(np.int64(fiberid)))
    long_filename = f"spec-{plate_string}-{mjd_string}-{fiberid_string}.fits"

    fig = plt.figure(figsize=(13, 9), constrained_layout=True)
    gs = GridSpec(2, 6, figure=fig, height_ratios=[1, 1], width_ratios=[1, 1, 1, 1, 1, 1])  # , hspace=0.3, wspace=0.5)
    ax1 = fig.add_subplot(gs[0, :2])  # LC
    ax2 = fig.add_subplot(gs[0, 2:4])  # Chi2 vs Amp plots OR Palerversa+2013 plot
    ax3 = fig.add_subplot(gs[0, 4:])  # CMD
    ax4 = fig.add_subplot(gs[1, :])  # spectra with lines
    fig.suptitle(f'RA= {ra_string} DEC= {dec_string}', fontsize=16)

    best_LC = vi.find_best_LC(ROW, logFAP_limit)

    lc_plot = vi.plot_LC_panel(ROW, ax1, best_LC, CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False)
    vi.plot_middle(ROW, TDSS_Table, best_LC, xiyizi, ax2, log10FAP=logFAP_limit)
    vi.plot_CMD(TDSSprop, prop_id, ax3)
    this_EqW = vi.plot_SDSSspec(ROW, spec_dir, ax4)

    return fig, [ax1, ax2, ax3, ax4], this_Viplot_filename
# plt.savefig(f"{Vi_plots_dir}{ra_string}{dec_string}_Vi.pdf", dpi=600, bbox_inches='tight')
# # plt.show()
# plt.clf()
# plt.close()

In [ ]:
# make_Vi_plot_index = np.arange(20)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    for ii, prop_id in enumerate(trange(len(TDSS_Table))):
        fig, [ax1, ax2, ax3, ax4], this_Viplot_filename = make_Vi_propID(prop_id, TDSS_Table, TDSSprop, logFAP_limit=-5)
        plt.savefig(f"{Vi_plots_dir}{this_Viplot_filename}")
        plt.clf()
        plt.close()

In [ ]:
start_index = 0
with Pool(os.cpu_count()-2) as pool:
    r = list(tqdm(pool.imap(make_Vi_propID_parallel, range(start_index, len(TDSS_Table))), total=len(TDSS_Table)-start_index))


In [86]:
vi.get_Vi_panels([0,1,2,3,4], TDSS_Table, datestr, copy=True, moveDir='/Users/benjaminroulston/Desktop/Vi/')

5it [00:00, 22.08it/s]


array(['/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-04/Vi_plots/0_000.01307-02.43008_Vi.pdf',
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-04/Vi_plots/1_000.01471+20.15233_Vi.pdf',
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-04/Vi_plots/2_000.02193+01.55998_Vi.pdf',
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-04/Vi_plots/3_000.02716+26.08195_Vi.pdf',
       '/Users/benjaminroulston/Dropbox/Research/TDSS/Variable_Stars/WORKING_DIRECTORY/Vi/2022-04-04/Vi_plots/4_000.03463+00.52496_Vi.pdf'],
      dtype='<U129')

In [8]:
logFAP_limit=-5
prop_id = 15497

ROW = TDSS_Table[prop_id]
object_ra = ROW['ra_GaiaEDR3']
object_dec = ROW['dec_GaiaEDR3']
ra_string = '{:0>9.5f}'.format(object_ra)
dec_string = '{:0=+9.5f}'.format(object_dec)

this_Viplot_filename = f"{prop_id}_{ra_string}{dec_string}_Vi.pdf"

mjd = ROW['mjd']
plate = ROW['plate']
fiberid = ROW['fiber']
mjd_string = '{:0>5}'.format(str(np.int64(mjd)))
plate_string = '{:0>4}'.format(str(np.int64(plate)))
fiberid_string = '{:0>4}'.format(str(np.int64(fiberid)))
long_filename = f"spec-{plate_string}-{mjd_string}-{fiberid_string}.fits"

isCSS = not np.ma.is_masked(ROW['CSS_Nepochs'])
isZTFg = not np.ma.is_masked(ROW['ZTF_g_Nepochs'])
isZTFr = not np.ma.is_masked(ROW['ZTF_r_Nepochs'])

LCs_name = np.array(['CSS', "ZTF_g", "ZTF_r"])
isLCs = [isCSS, isZTFg, isZTFr]
N_LCs = np.sum(isLCs)
selected_LCnames = LCs_name[np.where(isLCs)[0]]
print(N_LCs, isLCs)


%matplotlib inline
matplotlib.rcParams['axes.titlesize'] = 7


fig = plt.figure(figsize=(12, 12), constrained_layout=True, dpi=600)
gs0 = gridspec.GridSpec(3, 2, figure=fig, height_ratios=[2.0/4, 1./4, 1/4], width_ratios=[3/4, 1/4])

gs00 = gridspec.GridSpecFromSubplotSpec(N_LCs, 1, subplot_spec=gs0[0, 0], hspace=0.001)
gs01 = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs0[:, 1], hspace=0.0)
gs10 = gridspec.GridSpecFromSubplotSpec(1, N_LCs, subplot_spec=gs0[1, 0], hspace=0.0, height_ratios=[1.0])
gs20 = gridspec.GridSpecFromSubplotSpec(1, 1, subplot_spec=gs0[2, 0], hspace=0.0, height_ratios=[1])

if N_LCs==1:
    ax1 = fig.add_subplot(gs00[0, 0])
    
    ax4 = fig.add_subplot(gs10[0, 0])
elif N_LCs==2:
    if 'CSS' in selected_LCnames:
        ax1 = fig.add_subplot(gs00[0, 0])
        ax2 = fig.add_subplot(gs00[1, 0])
    else:
        ax1 = fig.add_subplot(gs00[0, 0])
        ax2 = fig.add_subplot(gs00[1, 0], sharex=ax1)
        plt.setp(ax1.get_xticklabels(), visible=False)
    
    ax4 = fig.add_subplot(gs10[0, 0])
    ax5 = fig.add_subplot(gs10[0, 1])
elif N_LCs==3:
    ax1 = fig.add_subplot(gs00[0, 0])
    ax2 = fig.add_subplot(gs00[1, 0])
    ax3 = fig.add_subplot(gs00[2, 0], sharex=ax2)
    plt.setp(ax2.get_xticklabels(), visible=False)
    
    ax4 = fig.add_subplot(gs10[0, 0])
    ax5 = fig.add_subplot(gs10[0, 1])
    ax6 = fig.add_subplot(gs10[0, 2])

ax10 = fig.add_subplot(gs20[0, 0])

ax11 = fig.add_subplot(gs01[0, 0])
ax12 = fig.add_subplot(gs01[1, 0])
ax13 = fig.add_subplot(gs01[2, 0])


best_LC = vi.find_best_LC(ROW, logFAP_limit)

if N_LCs==1:
    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax1, selected_LCnames[0], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=-np.inf)
    ax1.set_title("")
    ax1.set_ylabel(f"{selected_LCnames[0].replace('_',' ')} mag")
    ax1.legend(ax1.get_legend_handles_labels()[0], ax1.get_legend_handles_labels()[1], loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0))

    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax4, selected_LCnames[0], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=1.0)
    ax1.set_ylabel(f"{selected_LCnames[0].replace('_',' ')} mag")
    ax4.get_legend().remove()
elif N_LCs==2:
    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax1, selected_LCnames[0], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=-np.inf)
    ax1.set_title("")
    ax1.set_xlabel("")
    ax1.set_ylabel(f"{selected_LCnames[0].replace('_',' ')} mag")
    ax1.legend(ax1.get_legend_handles_labels()[0], ax1.get_legend_handles_labels()[1], loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0))

    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax4, selected_LCnames[0], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=1.0)
    ax4.set_ylabel(f"{selected_LCnames[0].replace('_',' ')} mag")
    ax4.get_legend().remove()
    
    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax2, selected_LCnames[1], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=-np.inf)
    ax2.set_title("")
    ax2.set_ylabel(f"{selected_LCnames[1].replace('_',' ')} mag")
    ax2.get_legend().remove()

    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax5, selected_LCnames[1], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=1.0)
    ax5.set_ylabel(f"{selected_LCnames[1].replace('_',' ')} mag")
    ax5.get_legend().remove()
elif N_LCs==3:
    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax1, selected_LCnames[0], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=-np.inf)
    ax1.set_title("")
    ax1.set_xlabel("")
    ax1.set_ylabel(f"{selected_LCnames[0].replace('_',' ')} mag")
    ax1.legend(ax1.get_legend_handles_labels()[0], ax1.get_legend_handles_labels()[1], loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0))

    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax4, selected_LCnames[0], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=1.0)
    ax4.set_ylabel(f"{selected_LCnames[0].replace('_',' ')} mag")
    ax4.get_legend().remove()
    
    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax2, selected_LCnames[1], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=-np.inf)
    ax2.set_title("")
    ax2.set_xlabel("")
    ax2.set_ylabel(f"{selected_LCnames[1].replace('_',' ')} mag")
    ax2.get_legend().remove()

    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax5, selected_LCnames[1], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=1.0)
    ax5.set_ylabel(f"{selected_LCnames[1].replace('_',' ')} mag")
    ax5.get_legend().remove()
    
    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax3, selected_LCnames[2], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=-np.inf)
    ax3.set_title("")
    ax3.set_ylabel(f"{selected_LCnames[2].replace('_',' ')} mag")
    ax3.get_legend().remove()

    lc_plot = vi.plot_LC_panel(ROW, TDSSprop, ax6, selected_LCnames[2], CSS_LC_dir, ZTF_g_LCs, ZTF_r_LCs, plot_rejected=False,  logFAPlimit=1.0)
    ax6.set_ylabel(f"{selected_LCnames[2].replace('_',' ')} mag")
    ax6.get_legend().remove()

vi.plot_middle(ROW, TDSS_Table, best_LC, xiyizi, ax11, one_log10_FAP=-np.inf, all_log10FAP=-10.0)
ax11.set_title(best_LC + "\n" + ax11.get_title())
vi.plot_middle(ROW, TDSS_Table, best_LC, xiyizi, ax12, one_log10_FAP=1.0, all_log10FAP=-10.0)
ax12.set_title(best_LC + "\n" + ax12.get_title())
vi.plot_CMD(TDSSprop, prop_id, ax13)
ax13.set_title("" + ax12.get_title())
this_EqW = vi.plot_SDSSspec(ROW, spec_dir, ax10)

plt.savefig("/Users/benjaminroulston/Desktop/test.pdf", dpi=600, bbox_inches='tight')

3 [True, True, True]


/Users/benjaminroulston/anaconda3/envs/Research/lib/python3.9/site-packages/numpy/ma/core.py:3377: UserWarning: Warning: converting a masked element to nan.
  _data[indx] = dval
/Users/benjaminroulston/anaconda3/envs/Research/lib/python3.9/site-packages/matplotlib/axes/_base.py:2480: UserWarning: Warning: converting a masked element to nan.
  xys = np.asarray(xys)
/Users/benjaminroulston/anaconda3/envs/Research/lib/python3.9/site-packages/matplotlib/cbook/__init__.py:1298: UserWarning: Warning: converting a masked element to nan.
  return np.asarray(x, float)
